In [68]:
#Imports and directory
import os
import pickle
import random
import math
import numpy as np
import pandas as pd
import matplotlib
import torch
import torch.nn as nn
from matplotlib import pyplot as plt
os.chdir("/home/ec2-user/CS-230-Deep-Learning-Project")
#os.chdir(r"C:\VScode\Projet Stanford CS230\CS-230-Deep-Learning-Project")
#os.chdir(r"C:\Users\gotta\OneDrive\Documents\Bureau\X\4A\US\Stanford\Classes\CS 230\Project\CS-230-Deep-Learning-Project")

In [69]:
# Basic functions including device and seeding
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print("USING DEVICE:", device)

def save_object_to_file(obj, filepath):
    with open(filepath, "wb") as f:
        pickle.dump(obj, f)

def read_object_from_file(filepath):
    with open(filepath, "rb") as f:
        return pickle.load(f)

def seed_everything(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


USING DEVICE: cuda:0


In [70]:
# --- Data preparation ---
DATA_CSV_FR = os.path.join("data", "FR_price", "FR_lmp_processed.csv")
DATA_CSV_LOAD = os.path.join("data", "FR_DAM_Load", "FR_load_processed.csv")
DATA_CSV_REN = os.path.join("data", "FR_REN", "FR_ren_processed.csv")
DATA_CSV_LOAD_RT=os.path.join("data", "FR_DAM_Load", "FR_rt_load_processed.csv")
DATA_CSV_REN_RT=os.path.join("data", "FR_REN", "FR_rt_ren_processed.csv")


In [71]:
df_FR = pd.read_csv(DATA_CSV_FR)
load_FR = pd.read_csv(DATA_CSV_LOAD)
ren_FR = pd.read_csv(DATA_CSV_REN)
all_data_DA = pd.merge(df_FR, load_FR, on=['Time', 'Timezone', 'Year', 'Month', 'Day', 'Hour'], how='inner')
all_data_DA = pd.merge(all_data_DA, ren_FR, on=['Time', 'Timezone', 'Year', 'Month', 'Day', 'Hour'], how='inner')
all_data_DA.rename(columns={'MW':'Load_DA','Wind Onshore':'Wind_DA','Solar':'Solar_DA'}, inplace=True)
load_FR_rt = pd.read_csv(DATA_CSV_LOAD_RT)
ren_FR_rt = pd.read_csv(DATA_CSV_REN_RT)
all_data_RT = pd.merge(df_FR, load_FR_rt, on=['Time', 'Timezone', 'Year', 'Month', 'Day', 'Hour'], how='inner')
all_data_RT = pd.merge(all_data_RT, ren_FR_rt, on=['Time', 'Timezone', 'Year', 'Month', 'Day', 'Hour'], how='inner')
all_data_RT.rename(columns={'MW':'Load_RT','Wind Onshore':'Wind_RT','Solar':'Solar_RT'}, inplace=True)

all_data=pd.merge(all_data_DA,all_data_RT,on=['Time', 'Timezone', 'Year', 'Month', 'Day', 'Hour','EUR/MWh'], how='inner')
all_data

,Time,Year,Month,Day,Hour,Timezone,EUR/MWh,Load_DA,Wind_DA,Solar_DA,Load_RT,Wind_RT,Solar_RT
0,2019-10-01 00:00:00+02:00,2019,10,1,0,+02:00,33.09,44450.0,4975.31,0.00,43062.00,6076.00,0.00
1,2019-10-01 01:00:00+02:00,2019,10,1,1,+02:00,27.72,41300.0,5338.58,0.00,40483.00,6137.00,0.00
2,2019-10-01 02:00:00+02:00,2019,10,1,2,+02:00,23.12,40050.0,5702.33,0.00,39207.00,6342.00,0.00
3,2019-10-01 03:00:00+02:00,2019,10,1,3,+02:00,16.46,37750.0,5667.94,0.00,37004.00,6355.00,0.00
4,2019-10-01 04:00:00+02:00,2019,10,1,4,+02:00,15.66,36600.0,5633.30,0.00,36399.00,6217.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
52142,2025-09-30 19:00:00+02:00,2025,9,30,19,+02:00,117.00,49000.0,2435.69,2158.72,49421.41,1939.43,901.85
52143,2025-09-30 20:00:00+02:00,2025,9,30,20,+02:00,106.18,50400.0,2929.19,315.89,50315.77,2445.66,244.45
52144,2025-09-30 21:00:00+02:00,2025,9,30,21,+02:00,83.23,46700.0,3480.47,0.00,47039.08,3116.56,0.00
52145,2025-09-30 22:00:00+02:00,2025,9,30,22,+02:00,73.00,44100.0,3879.11,0.00,45544.52,3422.12,0.00


In [72]:
#Convert hour, day, month into cyclic features
def add_cyclical_features(df):
    df['Date']=pd.to_datetime(df[['Year', 'Month', 'Day', 'Hour']])
    df["DayOfWeek"] = df['Date'].dt.dayofweek
    # Hour of day (0-23)
    df["sin_hour"] = np.sin(2 * np.pi * df["Hour"] / 24)
    df["cos_hour"] = np.cos(2 * np.pi * df["Hour"] / 24)

    # Day of week (0-6 or 1-7 depending on your data)
    df["sin_day"] = np.sin(2 * np.pi * df["DayOfWeek"] / 7)
    df["cos_day"] = np.cos(2 * np.pi * df["DayOfWeek"] / 7)

    # Month (1-12)
    df["sin_month"] = np.sin(2 * np.pi * df["Month"] / 12)
    df["cos_month"] = np.cos(2 * np.pi * df["Month"] / 12)
    return df
all_data = add_cyclical_features(all_data)

all_data


,Time,Year,Month,Day,Hour,Timezone,EUR/MWh,Load_DA,Wind_DA,Solar_DA,...,Wind_RT,Solar_RT,Date,DayOfWeek,sin_hour,cos_hour,sin_day,cos_day,sin_month,cos_month
0,2019-10-01 00:00:00+02:00,2019,10,1,0,+02:00,33.09,44450.0,4975.31,0.00,...,6076.00,0.00,2019-10-01 00:00:00,1,0.000000,1.000000,0.781831,0.62349,-0.866025,5.000000e-01
1,2019-10-01 01:00:00+02:00,2019,10,1,1,+02:00,27.72,41300.0,5338.58,0.00,...,6137.00,0.00,2019-10-01 01:00:00,1,0.258819,0.965926,0.781831,0.62349,-0.866025,5.000000e-01
2,2019-10-01 02:00:00+02:00,2019,10,1,2,+02:00,23.12,40050.0,5702.33,0.00,...,6342.00,0.00,2019-10-01 02:00:00,1,0.500000,0.866025,0.781831,0.62349,-0.866025,5.000000e-01
3,2019-10-01 03:00:00+02:00,2019,10,1,3,+02:00,16.46,37750.0,5667.94,0.00,...,6355.00,0.00,2019-10-01 03:00:00,1,0.707107,0.707107,0.781831,0.62349,-0.866025,5.000000e-01
4,2019-10-01 04:00:00+02:00,2019,10,1,4,+02:00,15.66,36600.0,5633.30,0.00,...,6217.00,0.00,2019-10-01 04:00:00,1,0.866025,0.500000,0.781831,0.62349,-0.866025,5.000000e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52142,2025-09-30 19:00:00+02:00,2025,9,30,19,+02:00,117.00,49000.0,2435.69,2158.72,...,1939.43,901.85,2025-09-30 19:00:00,1,-0.965926,0.258819,0.781831,0.62349,-1.000000,-1.836970e-16
52143,2025-09-30 20:00:00+02:00,2025,9,30,20,+02:00,106.18,50400.0,2929.19,315.89,...,2445.66,244.45,2025-09-30 20:00:00,1,-0.866025,0.500000,0.781831,0.62349,-1.000000,-1.836970e-16
52144,2025-09-30 21:00:00+02:00,2025,9,30,21,+02:00,83.23,46700.0,3480.47,0.00,...,3116.56,0.00,2025-09-30 21:00:00,1,-0.707107,0.707107,0.781831,0.62349,-1.000000,-1.836970e-16
52145,2025-09-30 22:00:00+02:00,2025,9,30,22,+02:00,73.00,44100.0,3879.11,0.00,...,3422.12,0.00,2025-09-30 22:00:00,1,-0.500000,0.866025,0.781831,0.62349,-1.000000,-1.836970e-16


In [73]:
#normalization
# Columns to exclude
exclude_cols = ["Time", "Timezone", "Year", "Month", "Day", "Hour", 'Date', 'DayOfWeek', 'cos_hour', 'sin_hour', 'cos_day', 'sin_day', 'cos_month', 'sin_month']

# Columns we want to normalize
norm_cols = [c for c in all_data.columns if c not in exclude_cols]

# Compute stats only on selected columns
means = all_data[norm_cols].mean()
medians = all_data[norm_cols].median()
stds  = all_data[norm_cols].std()

# Apply Gaussian normalization only to norm_cols
all_data_norm = all_data.copy()
all_data_norm[norm_cols] = (all_data[norm_cols] - means) / stds

all_data = all_data_norm

print(means)
print(stds)

EUR/MWh       104.796131
Load_DA     50508.327325
Wind_DA      4708.093065
Solar_DA     2291.111826
Load_RT     50465.382685
Wind_RT      4540.707101
Solar_RT     2209.322986
dtype: float64
EUR/MWh       110.919744
Load_DA     11085.904859
Wind_DA      3484.189322
Solar_DA     3486.011115
Load_RT     10901.326210
Wind_RT      3295.262375
Solar_RT     3266.016272
dtype: float64


In [74]:
save_path = os.path.join("data/Datasets_2", "normalization_stats.pkl")

with open(save_path, "wb") as f:
    pickle.dump({
        "means": means,
        "stds": stds,
        "medians": medians,
        "norm_cols": norm_cols
    }, f)

print(f"Saved normalization stats to: {save_path}")

Saved normalization stats to: data/Datasets_2/normalization_stats.pkl


In [75]:
def create_sliding_window(data_df, string, window_days=7, forecast_horizon=1):
    """
    Outputs five DataFrames:
    - X_df: past window (arrays) + date
    - y_df: target next-day (arrays) + date
    - y_day_df: previous-day same-hour baseline + date
    - y_week_df: previous-week same-hour baseline + date
    - pct_df: percent change vs last window value + date
    """


    series = data_df[string].values
    dates = pd.to_datetime(data_df["Date"])

    hours_per_day = 24
    window_size = window_days * hours_per_day
    eps = 1e-9

    X, y = [], []
    y_day_before, y_week_before = [], []
    y_pct = []
    y_dates = []

    n = len(series)

    for i in range(0, n - window_size - forecast_horizon + 1):

        # Input window
        X_window = series[i : i + window_size]
        # Forecast target
        y_target = series[i + window_size : i + window_size + forecast_horizon]

        # Target date
        date_target = dates[i + window_size]

        # Previous-day baseline
        start_prev_day = i + window_size - hours_per_day
        if start_prev_day >= 0:
            prev_day = series[start_prev_day : start_prev_day + forecast_horizon]
        else:
            prev_day = np.full(forecast_horizon, np.nan)

        # Previous-week baseline
        start_prev_week = i + window_size - 7 * hours_per_day
        if start_prev_week >= 0:
            prev_week = series[start_prev_week : start_prev_week + forecast_horizon]
        else:
            prev_week = np.full(forecast_horizon, np.nan)

        # Percent change wrt last hour in window
        pct = (y_target - X_window[-1]) / (X_window[-1] + eps)

        # Append everything
        X.append(X_window.astype(np.float32))
        y.append(y_target.astype(np.float32))
        y_day_before.append(prev_day.astype(np.float32))
        y_week_before.append(prev_week.astype(np.float32))
        y_pct.append(pct.astype(np.float32))
        y_dates.append(date_target)

    # ---- Build DataFrames ----
    X_df = pd.DataFrame({"X": X, "Date": y_dates})
    y_df = pd.DataFrame({"y": y, "Date": y_dates})
    y_day_df = pd.DataFrame({"y_day": y_day_before, "Date": y_dates})
    y_week_df = pd.DataFrame({"y_week": y_week_before, "Date": y_dates})
    pct_df = pd.DataFrame({"pct": y_pct, "Date": y_dates})

    return X_df, y_df, y_day_df, y_week_df, pct_df


In [76]:
#Create sliding windows
X_df, y_df, y_day_df, y_week_df, y_per_df   = create_sliding_window(all_data,'EUR/MWh', window_days=7, forecast_horizon=1)
X_load_DA_df, _, _, _, _ = create_sliding_window(all_data, string='Load_DA', window_days=7)
X_wind_DA_df, _, _, _,_ = create_sliding_window(all_data, string='Wind_DA', window_days=7)
X_solar_DA_df, _, _, _,_ = create_sliding_window(all_data, string='Solar_DA', window_days=7)
X_cos_hour_df, _, _, _, _ = create_sliding_window(all_data, string='cos_hour', window_days=7)
X_sin_hour_df, _, _, _,_ = create_sliding_window(all_data, string='sin_hour', window_days=7)
X_cos_day_df, _, _, _,_ = create_sliding_window(all_data, string='cos_day', window_days=7)
X_sin_day_df, _, _, _,_ = create_sliding_window(all_data, string='sin_day', window_days=7)
X_cos_month_df, _, _, _,_ = create_sliding_window(all_data, string='cos_month', window_days=7)
X_sin_month_df, _, _, _,_ = create_sliding_window(all_data, string='sin_month', window_days=7)

X_df = X_df.rename(columns={'X': 'EUR/MWh'})
X_load_DA_df = X_load_DA_df.rename(columns={'X': 'Load_DA'})
X_wind_DA_df= X_wind_DA_df.rename(columns={'X': 'Wind_DA'})
X_solar_DA_df = X_solar_DA_df.rename(columns={'X': 'Solar_DA'})
X_cos_hour_df = X_cos_hour_df.rename(columns={'X': 'cos_hour'})
X_sin_hour_df= X_sin_hour_df.rename(columns={'X': 'sin_hour'})
X_cos_day_df = X_cos_day_df.rename(columns={'X': 'cos_day'})
X_sin_day_df = X_sin_day_df.rename(columns={'X': 'sin_day'})
X_cos_month_df= X_cos_month_df.rename(columns={'X': 'cos_month'})
X_sin_month_df = X_sin_month_df.rename(columns={'X': 'sin_month'})


In [77]:
X_solar_DA_df

,Solar_DA,Date
0,"[-0.6572302, -0.6572302, -0.6572302, -0.657230...",2019-10-08 00:00:00
1,"[-0.6572302, -0.6572302, -0.6572302, -0.657230...",2019-10-08 01:00:00
2,"[-0.6572302, -0.6572302, -0.6572302, -0.657230...",2019-10-08 02:00:00
3,"[-0.6572302, -0.6572302, -0.6572302, -0.657230...",2019-10-08 03:00:00
4,"[-0.6572302, -0.6572302, -0.6572302, -0.657230...",2019-10-08 04:00:00
...,...,...
51974,"[0.75057083, -0.044819657, -0.5472994, -0.6571...",2025-09-30 19:00:00
51975,"[-0.044819657, -0.5472994, -0.6571441, -0.6572...",2025-09-30 20:00:00
51976,"[-0.5472994, -0.6571441, -0.6572302, -0.657230...",2025-09-30 21:00:00
51977,"[-0.6571441, -0.6572302, -0.6572302, -0.657230...",2025-09-30 22:00:00


In [78]:
#normalization



In [79]:
# Merge all relevant DataFrames on 'Date'
all_data_df = (
    X_df
    .merge(y_df, on='Date')
    .merge(X_load_DA_df, on='Date')
    .merge(X_wind_DA_df, on='Date')
    .merge(X_solar_DA_df, on='Date')
    .merge(X_cos_hour_df, on='Date')
    .merge(X_sin_hour_df, on='Date')
    .merge(X_cos_day_df, on='Date')
    .merge(X_sin_day_df, on='Date')
    .merge(X_cos_month_df, on='Date')
    .merge(X_sin_month_df, on='Date')
    .merge(y_day_df, on='Date')
    .merge(y_week_df, on='Date')
    .merge(y_per_df, on='Date')
)

# Reorder and select only the desired columns
all_data_df = all_data_df[['Date', 'EUR/MWh', 'Load_DA', 'Wind_DA', 'Solar_DA', 'cos_hour','sin_hour',	'cos_day','sin_day', 'cos_month',	'sin_month','y', 'y_day', 'y_week', 'pct']]
all_data_df

,Date,EUR/MWh,Load_DA,Wind_DA,Solar_DA,cos_hour,sin_hour,cos_day,sin_day,cos_month,sin_month,y,y_day,y_week,pct
0,2019-10-08 00:00:00,"[-0.6464686, -0.694882, -0.7363534, -0.7963968...","[-0.5464892, -0.8306338, -0.9433896, -1.150860...","[0.076694146, 0.18095657, 0.28535676, 0.275486...","[-0.6572302, -0.6572302, -0.6572302, -0.657230...","[1.0, 0.9659258, 0.8660254, 0.70710677, 0.5, 0...","[0.0, 0.25881904, 0.5, 0.70710677, 0.8660254, ...","[0.6234898, 0.6234898, 0.6234898, 0.6234898, 0...","[0.7818315, 0.7818315, 0.7818315, 0.7818315, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[-0.66855663],[-0.6834323],[-0.6464686],[0.0295726]
1,2019-10-08 01:00:00,"[-0.694882, -0.7363534, -0.7963968, -0.8036092...","[-0.8306338, -0.9433896, -1.1508603, -1.254595...","[0.18095657, 0.28535676, 0.27548644, 0.2655443...","[-0.6572302, -0.6572302, -0.6572302, -0.657230...","[0.9659258, 0.8660254, 0.70710677, 0.5, 0.2588...","[0.25881904, 0.5, 0.70710677, 0.8660254, 0.965...","[0.6234898, 0.6234898, 0.6234898, 0.6234898, 0...","[0.7818315, 0.7818315, 0.7818315, 0.7818315, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[-0.6929887],[-0.70074207],[-0.694882],[0.036544517]
2,2019-10-08 02:00:00,"[-0.7363534, -0.7963968, -0.80360925, -0.75195...","[-0.9433896, -1.1508603, -1.2545956, -1.14635,...","[0.28535676, 0.27548644, 0.26554438, 0.2556999...","[-0.6572302, -0.6572302, -0.6572302, -0.657230...","[0.8660254, 0.70710677, 0.5, 0.25881904, 6.123...","[0.5, 0.70710677, 0.8660254, 0.9659258, 1.0, 0...","[0.6234898, 0.6234898, 0.6234898, 0.6234898, 0...","[0.7818315, 0.7818315, 0.7818315, 0.7818315, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[-0.7194042],[-0.70137316],[-0.7363534],[0.038118217]
3,2019-10-08 03:00:00,"[-0.7963968, -0.80360925, -0.75195026, -0.5976...","[-1.1508603, -1.2545956, -1.14635, -0.7720008,...","[0.27548644, 0.26554438, 0.25569993, 0.2568508...","[-0.6572302, -0.6572302, -0.6572302, -0.657230...","[0.70710677, 0.5, 0.25881904, 6.123234e-17, -0...","[0.70710677, 0.8660254, 0.9659258, 1.0, 0.9659...","[0.6234898, 0.6234898, 0.6234898, 0.6234898, 0...","[0.7818315, 0.7818315, 0.7818315, 0.7818315, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[-0.7376156],[-0.71246225],[-0.7963968],[0.02531451]
4,2019-10-08 04:00:00,"[-0.80360925, -0.75195026, -0.59760445, -0.493...","[-1.2545956, -1.14635, -0.7720008, -0.21724229...","[0.26554438, 0.25569993, 0.25685084, 0.2580189...","[-0.6572302, -0.6572302, -0.6572302, -0.657230...","[0.5, 0.25881904, 6.123234e-17, -0.25881904, -...","[0.8660254, 0.9659258, 1.0, 0.9659258, 0.86602...","[0.6234898, 0.6234898, 0.6234898, 0.6234898, 0...","[0.7818315, 0.7818315, 0.7818315, 0.7818315, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[-0.75808084],[-0.7202156],[-0.80360925],[0.02774514]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101120,2025-09-30 19:00:00,"[-0.70615137, -0.4218918, -0.5784915, -0.77683...","[-0.44275388, -0.1901809, -0.08193533, -0.3345...","[0.82202965, 0.6571764, 0.5944731, 0.49475983,...","[0.75057083, -0.044819657, -0.5472994, -0.6571...","[-1.8369701e-16, 0.25881904, 0.5, 0.70710677, ...","[-1.0, -0.9659258, -0.8660254, -0.70710677, -0...","[0.6234898, 0.6234898, 0.6234898, 0.6234898, 0...","[0.7818315, 0.7818315, 0.7818315, 0.7818315, 0...","[-1.8369701e-16, -1.8369701e-16, -1.8369701e-1...","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....",[0.11002432],[-0.17847256],[-0.70615137],[-1.5596532]
101121,2025-09-30 20:00:00,"[-0.4218918, -0.5784915, -0.7768331, -0.746811...","[-0.1901809, -0.08193533, -0.3345083, -0.57806...","[0.6571764, 0.5944731, 0.49475983, 0.42686743,...","[-0.044819657, -0.5472994, -0.6571441, -0.6572...","[0.25881904

In [80]:
SEED = 42
seed_everything(SEED)

In [81]:
# --- Train, Dev, Test Split based on randomly splitting by month---
def month_based_split(df, train_ratio=0.6, dev_ratio=0.2, seed=42):

    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])

    # Create a unique month identifier
    df["YearMonth"] = df["Date"].dt.to_period("M")

    # List of unique months
    all_months = df["YearMonth"].unique()
    n_months = len(all_months)

    # Shuffle months
    rng = np.random.default_rng(seed)
    shuffled_months = rng.permutation(all_months)

    # Compute split sizes (in months)
    n_train = int(train_ratio * n_months)
    n_dev   = int(dev_ratio   * n_months)


    # Assign months
    train_months = shuffled_months[:n_train]
    dev_months   = shuffled_months[n_train : n_train + n_dev]
    test_months  = shuffled_months[n_train + n_dev :]

    # Build the splits
    train_df = df[df["YearMonth"].isin(train_months)].drop(columns="YearMonth")
    dev_df   = df[df["YearMonth"].isin(dev_months)].drop(columns="YearMonth")
    test_df  = df[df["YearMonth"].isin(test_months)].drop(columns="YearMonth")

    return train_df, dev_df, test_df, train_months, dev_months, test_months


In [82]:
train_df, dev_df, test_df, train_months, dev_months, test_months = month_based_split(all_data_df)

print("TRAIN months:", train_months)
print("DEV months:", dev_months)
print("TEST months:", test_months)

print(len(train_df), len(dev_df), len(test_df))


TRAIN months: [Period('2023-10', 'M') Period('2022-02', 'M') Period('2021-04', 'M')
 Period('2021-11', 'M') Period('2023-01', 'M') Period('2022-11', 'M')
 Period('2024-10', 'M') Period('2023-12', 'M') Period('2022-07', 'M')
 Period('2022-03', 'M') Period('2024-07', 'M') Period('2025-01', 'M')
 Period('2023-06', 'M') Period('2021-03', 'M') Period('2020-02', 'M')
 Period('2021-10', 'M') Period('2021-07', 'M') Period('2020-05', 'M')
 Period('2021-12', 'M') Period('2025-02', 'M') Period('2025-09', 'M')
 Period('2024-06', 'M') Period('2025-07', 'M') Period('2020-03', 'M')
 Period('2019-11', 'M') Period('2022-01', 'M') Period('2024-01', 'M')
 Period('2019-12', 'M') Period('2023-02', 'M') Period('2025-08', 'M')
 Period('2025-06', 'M') Period('2021-06', 'M') Period('2024-08', 'M')
 Period('2022-06', 'M') Period('2023-04', 'M') Period('2021-09', 'M')
 Period('2024-04', 'M') Period('2021-01', 'M') Period('2020-07', 'M')
 Period('2022-05', 'M') Period('2023-05', 'M') Period('2021-02', 'M')
 Perio

In [83]:
#Save train, dev, test sets
train_df.to_pickle(os.path.join("data", "Datasets_2", "train_set.pkl"))
dev_df.to_pickle(os.path.join("data", "Datasets_2", "dev_set.pkl"))
test_df.to_pickle(os.path.join("data", "Datasets_2", "test_set.pkl"))
#train_df.to_csv(os.path.join("data", "Datasets_2","train_set.csv"), index=False)
#dev_df.to_csv(os.path.join("data", "Datasets_2","dev_set.csv"), index=False)
#test_df.to_csv(os.path.join("data", "Datasets_2","test_set.csv"), index=False)